# Candidate Generation

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2
%reload_ext autoreload
import os
from pathlib import Path

# Add src directory to Python path
PROJECT_ROOT = Path.cwd().parent
# sys.path.append(str(PROJECT_ROOT / "src"))
os.chdir(PROJECT_ROOT)
import pandas as pd
import pickle
from src.candidate_retrieval.PreviousPurchases import PreviousPurchases
from src.candidate_retrieval.ItemSimilarity import ItemSimilarityCandidates
from src.candidate_retrieval.RecencyCandidates import RecencyCandidates
from src.candidate_retrieval.AlsCandidates import ALSCandidates
from src.candidate_retrieval.StyleAlternatives import SameProductColorCandidates
from src.candidate_retrieval.CandidateRetrievalPipeline import CandidateGeneratorEnsemble
from src.utils.generate_labels import create_labels_with_val_set

ModuleNotFoundError: No module named 'src'

In [1]:
cleaned_df_folder = "../input/cleaned"
def load_from_parquet(cleaned_df_name):
    print("Loading data from Parquet...")
    return pd.read_parquet(cleaned_df_folder + "/" + cleaned_df_name + ".parquet")

In [ ]:
cols_needed = ['t_dat', 'customer_id', 'article_id','week']

train_transactions = pd.read_parquet(
    "../input/cleaned/train_transactions.parquet", 
    columns=cols_needed
)

NameError: name 'pd' is not defined

In [ ]:

val_transactions = load_from_parquet("val_transactions")
articles = load_from_parquet("articles")

In [ ]:
val_customers = val_transactions['customer_id'].unique()

generators_config = {
    "previous_purchases": (PreviousPurchases, {"lookback_weeks": 12, "k": 15}),
    "item_similarity": (
        ItemSimilarityCandidates,
        {
            "lookback_weeks": 6,
            "min_item_frequency": 30,
            "neighbors": 10,
            "k": 25
        },
    ),
    "recency": (RecencyCandidates, {"lookback_weeks": 12, "k": 20}),
    "als": (ALSCandidates, {"lookback_weeks": 12, "k": 40}),
    "same_product_color": (SameProductColorCandidates, {"lookback_weeks": 12, "k": 20})
}

# Initialize the Ensemble
ensemble = CandidateGeneratorEnsemble(
    transactions=train_transactions, articles=articles, generators=generators_config
)

# Generate candidates for validation customers
print(
    f"Generating candidates via ensemble for {len(val_customers):,} validation"
    " customers..."
)
recs_df = ensemble.generate(users=val_customers)

print(f"Generated {len(recs_df):,} total candidate rows.")

In [ ]:

recs_df.to_parquet(cleaned_df_folder + "/candidates_train.parquet", index=False)

In [ ]:
labelled_candidates = create_labels_with_val_set(recs_df, val_transactions)
labelled_candidates.to_parquet(cleaned_df_folder + "/labelled_candidates_train.parquet", index=False)